# Diplomado de Grado en Análisis de Datos Aplicado a la Toma de Decisiones
## UNICOLOMBO — Fundación Universitaria Colombo Internacional · Cartagena
**Módulo 2:** Preparación y Análisis Exploratorio de Datos  
**Sesión 2:** Limpieza de Datos  sisii
**Docente:** Ing. Heyder Medrano Olier | **Período:** 2026 — Vacaciones

---
**Tipo:** Notebook de Actividad  
**Entrega:** 28 de junio de 2026 · 23:59 (hora Colombia)  
**Valor:** 5.0 puntos  

**Estudiante:** [Jefferson Arrieta Duran]  

> Aplique el proceso completo de limpieza al dataset DataRetail LATAM.
> Cada decisión debe estar justificada en celdas Markdown.
> Al finalizar: *Kernel → Restart & Run All* y verificar que no haya errores.

## 1. Librerías

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import missingno as msno, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('OK')

## 2. Cargar Dataset

In [ ]:
import numpy as np, pandas as pd, random
from datetime import datetime, timedelta
np.random.seed(42); random.seed(42)
N=2000
ciudades=["Bogotá","Medellín","Cali","Barranquilla","Cartagena",
          "Lima","Ciudad de México","Buenos Aires","Santiago","Montevideo"]
ciu_s=ciudades+["bogota","BOGOTÁ","Medellin","cali ","barranquilla","SANTIAGO"]
paises=["Colombia","Perú","México","Argentina","Chile","Uruguay"]
cats=["Computación","Periféricos","Audio","Telefonía","Accesorios","Gaming","Componentes"]
canales=["Tienda Física","E-commerce","Distribuidor","Corporativo","Marketplace"]
can_s=canales+["tienda fisica","E-Comerce","distribuidor ","CORPORATIVO"]
prods=["Laptop Pro","Tablet Ultra","Monitor 4K","Teclado Inalámbrico",
       "Auriculares BT","Parlante Portátil","Smartphone Galaxy",
       "Cámara Web HD","Smartwatch Pro","Mouse Gaming","SSD 1TB","GPU RTX 4060"]
precios={"Laptop Pro":1200,"Tablet Ultra":450,"Monitor 4K":380,"Teclado Inalámbrico":75,
         "Auriculares BT":120,"Parlante Portátil":90,"Smartphone Galaxy":680,
         "Cámara Web HD":95,"Smartwatch Pro":250,"Mouse Gaming":65,"SSD 1TB":130,"GPU RTX 4060":580}
rows=[]
for i in range(N):
    prod=random.choice(prods); cat=cats[prods.index(prod)%len(cats)]
    pais=random.choice(paises)
    ciudad=random.choice(ciu_s) if random.random()<0.15 else random.choice(ciudades)
    canal=random.choice(can_s) if random.random()<0.10 else random.choice(canales)
    qty=random.randint(1,20); precio=round(precios[prod]*np.random.uniform(0.8,1.3),2)
    desc=round(random.choice([0,0,0,0.05,0.10,0.15,0.20,0.30,0.35,0.50]),2)
    total=round(qty*precio*(1-desc),2); margen=round(total*np.random.uniform(-0.05,0.35),2)
    fecha=datetime(2024,1,1)+timedelta(days=random.randint(0,729))
    rows.append({"id_venta":f"V{i+1:05d}","fecha_venta":fecha.strftime("%Y-%m-%d"),
                 "id_cliente":f"C{random.randint(1,500):04d}",
                 "nombre_cliente":f"Cliente {random.randint(1,500)}",
                 "ciudad":ciudad,"pais":pais,"id_producto":f"P{prods.index(prod)+1:02d}",
                 "nombre_producto":prod,"categoria":cat,"canal_venta":canal,
                 "cantidad":qty,"precio_unitario":precio,"descuento":desc,
                 "total_venta":total,"margen_utilidad":margen})
df_raw=pd.DataFrame(rows)
idx_p=np.random.choice(df_raw.index,size=int(N*0.06),replace=False)
idx_t=np.random.choice(df_raw.index,size=int(N*0.04),replace=False)
idx_m=np.random.choice(df_raw.index,size=int(N*0.03),replace=False)
df_raw.loc[idx_p,"precio_unitario"]=np.nan
df_raw.loc[idx_t,"total_venta"]=np.nan
df_raw.loc[idx_m,"margen_utilidad"]=np.nan
dup_idx=np.random.choice(df_raw.index,size=40,replace=False)
df_raw=pd.concat([df_raw,df_raw.loc[dup_idx].copy()],ignore_index=True)
out_idx=np.random.choice(df_raw.index,size=10,replace=False)
df_raw.loc[out_idx,"total_venta"]=np.random.uniform(50000,200000,10)
df_raw=df_raw.sample(frac=1,random_state=42).reset_index(drop=True)
df=df_raw.copy()
df["fecha_venta"]=pd.to_datetime(df["fecha_venta"],errors="coerce")
df["precio_unitario"]=pd.to_numeric(df["precio_unitario"],errors="coerce")
print(f"Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Nulos: {df.isna().sum().sum():,} | Duplicados: {df.duplicated().sum():,}")
df.head(3)


## 3. Registrar Estado Inicial

Antes de cualquier modificación, registre los conteos de problemas detectados en S01.

In [ ]:
ri = {'filas':len(df),'nulos_precio':df['precio_unitario'].isna().sum(),
      'nulos_total':df['total_venta'].isna().sum(),'nulos_margen':df['margen_utilidad'].isna().sum(),
      'duplicados':df.duplicated().sum(),'desc_viol':(df['descuento']>0.30).sum()}
print('ESTADO INICIAL (antes de limpiar):')
for k,v in ri.items(): print(f'  {k:<20}: {v:,}')

## 4. Corrección de Tipos

>
Antes de realizar cualquier proceso de limpieza es necesario corregir los tipos de datos, ya que muchas operaciones dependen de que cada columna tenga el formato adecuado. En este caso, las columnas cantidad y descuento se convierten a valores numéricos utilizando pd.to_numeric(), lo que permite identificar automáticamente los datos inválidos y convertirlos en valores nulos (NaN) mediante el parámetro errors='coerce'. Además, la columna cantidad se transforma al tipo Int64, que permite almacenar números enteros y, al mismo tiempo, conservar valores faltantes.

In [ ]:
df['cantidad']  = pd.to_numeric(df['cantidad'],  errors='coerce').astype('Int64')
df['descuento'] = pd.to_numeric(df['descuento'], errors='coerce')
print('Tipos corregidos:', df.dtypes[['cantidad','precio_unitario','descuento','total_venta']].to_dict())

### 📝 Justificación
> [¿qué problema específico resuelve la corrección de tipos?]                  La corrección de tipos resuelve el problema de que los datos sean interpretados de forma incorrecta por Python. Por ejemplo, un valor numérico almacenado como texto no puede utilizarse para realizar cálculos, obtener estadísticas, detectar valores atípicos o imputar valores faltantes. Al convertir las columnas al tipo de dato adecuado, se garantiza que las operaciones del proceso de limpieza se ejecuten correctamente y que los resultados sean confiables.

## 5. Eliminación de Duplicados

In [ ]:
n0 = len(df)
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print(f'Eliminados: {n0-len(df):,} | Quedan: {len(df):,}')

In [ ]:
# Registros antes
print("Registros antes:", len(df))

# Eliminar duplicados exactos
df = df.drop_duplicates()

print("Registros después de eliminar duplicados exactos:", len(df))


In [ ]:
# Eliminar duplicados parciales por ID + fecha
df = df.drop_duplicates(subset=['id_venta', 'fecha_venta'], keep='first')

print("Registros después de eliminar duplicados por ID + fecha:", len(df))

### 📝 Justificación del `keep='first'`
> [¿Por qué esta estrategia? ¿Cuándo usaría `keep=False`?]
> Se utilizó la estrategia keep='first' porque permite conservar el primer registro de cada grupo de duplicados y eliminar únicamente las copias repetidas. En este caso, si existen varias filas con el mismo id_venta y fecha_venta, se asume que el primer registro es el válido y los demás corresponden a duplicados que deben eliminarse. La opción keep=False se utilizaría cuando no sea posible determinar cuál de los registros duplicados es el correcto o cuando se requiera eliminar todos los registros duplicados para realizar una revisión manual. Esta estrategia es útil en situaciones donde los duplicados pueden contener información diferente o existir inconsistencias que deban analizarse antes de conservar un registro.


## 6. Imputación de Nulos

Aplique las tres estrategias vistas en clase y justifique cada una.

In [ ]:
df.isnull().sum()


In [ ]:
# Imputación con la media
df['margen_utilidad'] = df['margen_utilidad'].fillna(df['margen_utilidad'].mean())

>Se utilizó la media para imputar los valores faltantes de **margen_utilidad** porque es una variable numérica continua. Esta estrategia reemplaza los valores nulos por el promedio de la columna, permitiendo conservar todos los registros sin alterar significativamente la distribución de los datos cuando los valores faltantes corresponden a un patrón MCAR (Missing Completely At Random).


In [ ]:
# Imputación con la mediana
df['total_venta'] = df['total_venta'].fillna(df['total_venta'].median())

>Se empleó la mediana para imputar los valores faltantes de **total_venta** porque esta variable puede contener valores extremos que influyen en el promedio. La mediana representa mejor el valor central y proporciona una estimación más robusta cuando existen outliers o una distribución asimétrica. Esta estrategia es apropiada para datos faltantes clasificados como MAR (Missing At Random).



In [ ]:
# Ejemplo de imputación con la moda
df['ciudad'] = df['ciudad'].fillna(df['ciudad'].mode()[0])

In [ ]:
df['categoria'] = df['categoria'].fillna(df['categoria'].mode()[0])
df['canal_venta'] = df['canal_venta'].fillna(df['canal_venta'].mode()[0])

>La imputación mediante la moda se utiliza en variables categóricas porque reemplaza los valores faltantes con la categoría que aparece con mayor frecuencia.  las variables categóricas no presentaron valores nulos. No obstante, se incluye como ejemplo porque es una de las estrategias de imputación estudiadas y resulta adecuada para este tipo de variables.



### 6.1 precio_unitario — Mediana por Categoría

In [ ]:
n0 = df['precio_unitario'].isna().sum()
med_cat = df.groupby('categoria')['precio_unitario'].transform('median')
df['precio_unitario'] = df['precio_unitario'].fillna(med_cat)
df['precio_unitario'] = df['precio_unitario'].fillna(df['precio_unitario'].median())
print(f'Nulos precio: {n0} → {df["precio_unitario"].isna().sum()}')

### 6.2 total_venta — Recálculo

In [ ]:
n0 = df['total_venta'].isna().sum()
mask = df['total_venta'].isna()
df.loc[mask,'total_venta'] = (
    df.loc[mask,'cantidad'] * df.loc[mask,'precio_unitario'] * (1-df.loc[mask,'descuento'])
).round(2)
print(f'Nulos total: {n0} → {df["total_venta"].isna().sum()}')

### 6.3 margen_utilidad — Mediana por Categoría

In [ ]:
n0 = df['margen_utilidad'].isna().sum()
med_m = df.groupby('categoria')['margen_utilidad'].transform('median')
df['margen_utilidad'] = df['margen_utilidad'].fillna(med_m)
df['margen_utilidad'] = df['margen_utilidad'].fillna(df['margen_utilidad'].median())
print(f'Nulos margen: {n0} → {df["margen_utilidad"].isna().sum()}')
print(f'Nulos totales: {df.isna().sum().sum():,}')

## 7. Visualización Missingno — Antes vs Después

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,4))
fig.suptitle('Nulos: Antes vs Después', fontweight='bold')
df_v = df_raw.copy()
df_v['precio_unitario']=pd.to_numeric(df_v['precio_unitario'],errors='coerce')
msno.bar(df_v, ax=axes[0], color='#e15759', fontsize=9)
axes[0].set_title('ANTES')
msno.bar(df, ax=axes[1], color='#59a14f', fontsize=9)
axes[1].set_title('DESPUÉS')
plt.tight_layout(); plt.show()

## 8. Tratamiento de Outliers — Capping IQR

In [ ]:
def cap_iqr(serie, nombre):
    Q1,Q3 = serie.quantile(.25), serie.quantile(.75)
    IQR=Q3-Q1; n=((serie<Q1-1.5*IQR)|(serie>Q3+1.5*IQR)).sum()
    print(f'{nombre}: {n} outliers capeados')
    return serie.clip(Q1-1.5*IQR, Q3+1.5*IQR)

df['total_venta']     = cap_iqr(df['total_venta'],     'total_venta')
df['precio_unitario'] = cap_iqr(df['precio_unitario'], 'precio_unitario')

### 📝 Capping vs Eliminación
> [¿Cuándo es preferible capear y cuándo eliminar el outlier? Dé un ejemplo con estos datos.]
>Es mejor aplicar **capping** cuando el valor extremo puede ser una venta real y no un error en los datos. De esta forma, el registro se conserva, pero se reduce su impacto en los análisis estadísticos. Por ejemplo, en este dataset una venta con un **`total_venta`** muy alto puede deberse a que un cliente compró una gran cantidad de productos, por lo que no sería correcto eliminar ese registro.
>En cambio, eliminar un outlier es más recomendable cuando se identifica que el dato es un error de registro. Por ejemplo, si un **`precio_unitario`** fuera negativo o tuviera un valor exageradamente alto por un error al ingresar la información, lo mejor sería eliminarlo o corregirlo, ya que no representa una situación real y podría afectar los resultados del análisis.
>Al aplicar la técnica de Winsorizing se identificaron y capearon **164 valores atípicos en la variable `total_venta`** y **101 en `precio_unitario`**. Esto indica que ambas variables presentaban una cantidad importante de valores extremos que podían influir en los resultados del análisis. En lugar de eliminar estos registros, se decidió limitar su impacto porque es posible que correspondan a ventas reales con montos o precios superiores al promedio. De esta manera, se conserva la información del dataset y se reduce la influencia de los valores extremos, obteniendo un conjunto de datos más estable para los análisis posteriores.



## 9. Normalización Categórica

In [ ]:
df['ciudad'] = df['ciudad'].str.strip().str.title()
mapa = {'Bogota':'Bogotá','Ciudad De Mexico':'Ciudad de México'}
df['ciudad'] = df['ciudad'].replace(mapa)
df['canal_venta'] = df['canal_venta'].str.strip()
mapa_c = {'tienda fisica':'Tienda Física','E-Comerce':'E-commerce',
          'distribuidor ':'Distribuidor','CORPORATIVO':'Corporativo'}
df['canal_venta'] = df['canal_venta'].replace(mapa_c)
print(f'Ciudades únicas: {df["ciudad"].nunique()} | Canales únicos: {df["canal_venta"].nunique()}')

In [ ]:
print(df['ciudad'].unique())
print(df['categoria'].unique())
print(df['canal_venta'].unique())

In [ ]:
# Eliminar espacios y convertir a mayúsculas
df['ciudad'] = df['ciudad'].str.strip().str.upper()
df['categoria'] = df['categoria'].str.strip().str.upper()
df['canal_venta'] = df['canal_venta'].str.strip().str.upper()

# Mapeo de valores inconsistentes
map_ciudad = {
    'MEDELLIN': 'MEDELLÍN'
}

map_canal = {
    'DISTRIBUIDOR': 'DISTRIBUIDOR'
}

df['ciudad'] = df['ciudad'].replace(map_ciudad)
df['canal_venta'] = df['canal_venta'].replace(map_canal)

# Verificar resultado
print(df['ciudad'].unique())
print(df['categoria'].unique())
print(df['canal_venta'].unique())

Se estandarizaron las columnas **ciudad**, **categoría** y **canal_venta** eliminando espacios en blanco, convirtiendo todos los valores a mayúsculas y corrigiendo algunas diferencias de escritura mediante un mapeo. Por ejemplo, en la columna **ciudad** aparecían los valores **"Medellín"** y **"Medellin"**, que representan la misma ciudad, por lo que se unificaron en un solo valor. De igual forma, en **canal_venta** se encontraron registros como **"Distribuidor"** y **"distribuidor"**, los cuales quedaron estandarizados como **"DISTRIBUIDOR"**. Este proceso mejora la calidad de los datos y evita que una misma categoría sea interpretada como si fueran valores diferentes durante el análisis.


## 10. Corrección de Dominio

In [ ]:
n_desc = (df['descuento']>0.30).sum()
df['descuento'] = df['descuento'].clip(upper=0.30)
df['total_venta'] = (df['cantidad']*df['precio_unitario']*(1-df['descuento'])).round(2)
print(f'Descuentos > 30% corregidos: {n_desc}')

En esta etapa se corrigieron inconsistencias relacionadas con los descuentos y el cálculo del valor total de las ventas. Se identificaron registros con descuentos superiores al 30 %, los cuales no cumplen con la política comercial definida para el análisis. Por esta razón, dichos valores fueron limitados a un máximo del 30 % utilizando la función `clip()`.

Además, se recalculó la variable **total_venta** a partir de la cantidad de productos, el precio unitario y el descuento aplicado. Esto permitió garantizar que el valor total de cada venta fuera coherente con la información registrada en las demás columnas y evitar diferencias ocasionadas por errores de cálculo o captura de datos.

Se identificaron y corrigieron 411 registros con descuentos superiores al 30 %. Posteriormente, se recalculó el valor de **total_venta** para todos los registros, garantizando que los montos finales fueran consistentes con la cantidad de productos vendidos, el precio unitario y el descuento aplicado. Estas correcciones contribuyen a mejorar la confiabilidad de la información para su posterior análisis.



In [ ]:
(df['margen_utilidad']<0).sum()

In [ ]:

# Corregir márgenes negativos
n_margen = (df['margen_utilidad'] < 0).sum()
df['margen_utilidad'] = df['margen_utilidad'].clip(lower=0)

# Recalcular total de venta
df['total_venta'] = (
    df['cantidad'] * df['precio_unitario'] * (1 - df['descuento'])
).round(2)

print(f'Descuentos > 30% corregidos: {n_desc}')
print(f'Márgenes negativos corregidos: {n_margen}')

Se corrigieron las inconsistencias encontradas en el dataset para mejorar la calidad de la información. En primer lugar, los descuentos mayores al 30 % se limitaron a ese valor porque la empresa estableció ese porcentaje como el máximo permitido. Además, se identificaron **243 registros con margen de utilidad negativo**, los cuales se ajustaron a cero para evitar valores que afectaran el análisis. Finalmente, se recalculó la columna **total_venta** utilizando la cantidad, el precio unitario y el descuento corregido, garantizando que los valores fueran consistentes con la información registrada.


## 11. Panel Visual

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(14,9))
fig.suptitle('Panel Limpieza S02 — Antes vs Después', fontweight='bold')
pd.to_numeric(df_raw['total_venta'],errors='coerce').dropna().hist(ax=axes[0,0],bins=50,color='#e15759',ec='white')
axes[0,0].set_title('total_venta ANTES')
df['total_venta'].hist(ax=axes[0,1],bins=50,color='#59a14f',ec='white')
axes[0,1].set_title('total_venta DESPUÉS')
df_raw['ciudad'].value_counts().head(8).plot(kind='bar',ax=axes[1,0],color='#e15759',ec='white')
axes[1,0].set_title('ciudad ANTES'); axes[1,0].tick_params(axis='x',rotation=45)
df['ciudad'].value_counts().head(8).plot(kind='bar',ax=axes[1,1],color='#59a14f',ec='white')
axes[1,1].set_title('ciudad DESPUÉS'); axes[1,1].tick_params(axis='x',rotation=45)
plt.tight_layout(); plt.show()

## 12. Comparación Final y Score

In [ ]:
rf = {'filas':len(df),'nulos_precio':df['precio_unitario'].isna().sum(),
      'nulos_total':df['total_venta'].isna().sum(),'nulos_margen':df['margen_utilidad'].isna().sum(),
      'duplicados':df.duplicated().sum(),'desc_viol':(df['descuento']>0.30).sum()}
print(f'{"Métrica":<20} {"Antes":>8} {"Después":>9}')
print('-'*42)
for k in rf:
    print(f'{k:<20} {ri.get(k,0):>8,} {rf[k]:>9,}')

s = df['total_venta']
Q1,Q3=s.quantile(.25),s.quantile(.75)
po=((s<Q1-1.5*(Q3-Q1))|(s>Q3+1.5*(Q3-Q1))).mean()*100
score=100-min(df.isna().mean().mean()*300,30)-min(po*1.5,15)
print(f'Score: {score:.1f}/100')

## 13. Exportar Dataset Limpio

In [ ]:
df.to_csv('DataRetail_LATAM_S02_limpio.csv',index=False,encoding='utf-8-sig')
print(f'Exportado: {len(df):,} filas x {df.shape[1]} columnas')

## 14. Reflexiones y Conclusiones

### Mis 5 hallazgos más importantes:

1. [Se eliminaron los registros duplicados, lo que permitió evitar que una misma venta se contabilizara más de una vez y mejoró la confiabilidad de la información.]
2. [Se identificaron valores nulos en las columnas total_venta y margen_utilidad, los cuales fueron imputados para conservar todos los registros y evitar la pérdida de información.]
3. [Se detectaron 164 valores atípicos en total_venta y 101 en precio_unitario, los cuales fueron tratados mediante Winsorizing para reducir su impacto sin eliminar los registros.]
4. [Se encontraron inconsistencias en variables de texto, como diferentes formas de escribir Medellín y Distribuidor, las cuales fueron estandarizadas para mantener un formato uniforme en todo el dataset.]
5. [Se corrigieron inconsistencias en los datos, como descuentos superiores al 30 %, márgenes de utilidad negativos y valores de total_venta que no coincidían con el cálculo esperado, dejando un conjunto de datos más consistente y listo para el análisis.]

### Impacto de negocio:
> [¿Qué problemas de negocio resuelve este dataset limpio para DataRetail LATAM?]

>Contar con un dataset limpio permite que DataRetail LATAM tome decisiones basadas en información más confiable. Al eliminar duplicados, corregir inconsistencias, tratar valores atípicos y completar los datos faltantes, los reportes de ventas reflejan de mejor manera la realidad del negocio. Esto ayuda a evaluar correctamente el desempeño de los productos, calcular ingresos y utilidades con mayor precisión, identificar tendencias de compra y diseñar estrategias comerciales con menor riesgo de tomar decisiones basadas en información incorrecta.


## Referencias
1. McKinney, W. (2022). *Python for Data Analysis* (3.ª ed.). O'Reilly.
2. Little, R. & Rubin, D. (2019). *Statistical Analysis with Missing Data* (3.ª ed.). Wiley.
3. García, S. et al. (2015). *Data Preprocessing in Data Mining*. Springer.
4. DAMA International (2017). *DAMA-DMBOK* (2.ª ed.).
5. Bilogur, A. (2023). missingno. https://github.com/ResidentMario/missingno

---
*Notebook Actividad S02 — UNICOLOMBO · 2026*